# 习题数值复算

按题号查找复算单元。首次使用时先运行安装单元（已有 `mpmath` 可跳过），再运行公共设置；之后可任选题号独立复算。

## 安装依赖

首次运行安装 mpmath；若环境已有，可跳过。

In [ ]:
%pip install mpmath

## 公共设置

以 40 位工作精度计算。零点按正虚部递增生成并在当前会话缓存；首次计算 1000 个零点可能较慢。这些近似值不构成区间认证。

In [ ]:
import mpmath as mp

mp.mp.dps = 40
theta = mp.siegeltheta

def theta_prime(t):
    z = mp.mpf(1)/4 + mp.j*t/2
    return (mp.re(mp.digamma(z)) - mp.log(mp.pi))/2

_gamma_cache = []
def zero_heights(N):
    for k in range(len(_gamma_cache)+1,N+1):
        _gamma_cache.append(mp.im(mp.zetazero(k)))
    return _gamma_cache[:N]

def completed_xi(s):
    if s == 1 or s == 0:
        return mp.mpf('0.5')
    return s*(s-1)*mp.pi**(-s/2)*mp.gamma(s/2)*mp.zeta(s)/2

def mobius(n):
    value = 1
    for p in range(2,n+1):
        if n % p == 0:
            n //= p
            if n % p == 0:
                return 0
            value = -value
    return value

def trivial_h(y):
    return mp.quad(lambda t: 1/(t*(t*t-1)*mp.log(t)), [y,2*y,mp.inf])

def explicit_components(L, gammas, M):
    return (mp.ei(L),
            -2*mp.fsum(mp.re(mp.ei((mp.mpf('0.5')+mp.j*g)*L)) for g in gammas),
            -mp.fsum(mp.ei(-2*k*L) for k in range(1,M+1)),
            -mp.log(2))

# 第一部分　素数计数的积分表示

## 1.1

使用题设给出的素数计数；以 li(x)−li(2) 计算偏移对数积分。输出各近似及带符号误差。

In [ ]:
A = mp.mpf('1.08366')
for x, pi_x in [(10**4,1229),(10**6,78498),(10**8,5761455),
                (10**12,37607912018),(10**18,24739954287740860)]:
    L = mp.log(x)
    values = [x/(L-1), x/(L-A), mp.li(x)-mp.li(2)]
    print(x, [mp.nstr(v, 18) for v in values],
          [mp.nstr(pi_x-v, 12) for v in values])

## 1.2

在 x=10⁶ 逐项累加渐近级数，比较最小项编号与实际误差最小的截断编号。这里用未偏移的 li。

In [ ]:
x = 10**6
L = mp.log(x)
target = mp.li(x)
term, partial = x/L, mp.mpf(0)
rows = []
for n in range(1, 26):
    partial += term
    rows.append((n, term, partial-target))
    if n in (1,4,8,12,13,14,15,18,20,25):
        print(n, mp.nstr(term, 8), mp.nstr(partial-target, 8))
    term *= n/L
print("least term:", min(rows,key=lambda r:r[1])[0])
print("least absolute error:", min(rows,key=lambda r:abs(r[2]))[0])

## 1.3

由 x=6.5 开始做三次 Newton 迭代，并复算解答给出的两侧符号。

In [ ]:
def D(x):
    return mp.li(x)-mp.li(2)-x/mp.log(x)
x = mp.mpf('6.5')
for j in range(3):
    x -= D(x)*mp.log(x)**2
    print(j+1, mp.nstr(x, 12), mp.nstr(D(x), 8))
for x in map(mp.mpf, ('6.57965','6.57975')):
    print(x, mp.nstr(D(x), 8))

## 3.4

取 x=10、T=10，复算三条竖线的积分，并核对向左移动时的两条水平边贡献。

In [ ]:
x, T = 10, 10
def kernel(s):
    return x**s/s
def vertical(c):
    return mp.quad(lambda t: mp.re(kernel(c+mp.j*t)), [0,T])/mp.pi
for c in (2,1,mp.mpf('0.5')):
    print(c, mp.nstr(vertical(c), 12))
top = mp.quad(lambda c: kernel(c+mp.j*T), [2,1])/(2*mp.pi*mp.j)
bottom = mp.quad(lambda c: kernel(c-mp.j*T), [1,2])/(2*mp.pi*mp.j)
print(mp.nstr(vertical(2)-vertical(1), 12), mp.nstr(-top-bottom, 12))

## 4.3

对比参数 0.1 的直接截断与参数 10 的模反演；分别输出截至第七、八项的误差。

In [ ]:
q = mp.mpf('0.1')
def theta_sum(y, n):
    return 1+2*mp.fsum(mp.exp(-mp.pi*y*k*k) for k in range(1,n+1))
reflected = theta_sum(1/q, 1)/mp.sqrt(q)
for n in (7,8):
    print(n, mp.nstr(reflected-theta_sum(q,n), 10))
print(mp.nstr(reflected, 18), mp.nstr(reflected-1/mp.sqrt(q), 10))

## 5.2

对 P=10、100、1000 计算 Euler 部分乘积、实际误差及平方倒数级数尾和。

In [ ]:
from math import isqrt
def is_prime(n):
    return n >= 2 and all(n%d for d in range(2,isqrt(n)+1))
for P in (10,100,1000):
    product = mp.fprod(1/(1-mp.mpf(p)**-2) for p in range(2,P+1) if is_prime(p))
    print(P, *(mp.nstr(v, 12) for v in (product,mp.zeta(2)-product,mp.zeta(2,P+1))))

# 第二部分　ζ 的解析延拓与函数方程

## 7.4

对照正整数阶乘的 Stirling 近似，并在 $t=1,10$ 比较 Gamma 模平方及其主项。

In [ ]:
for n in (1, 5, 10, 20):
    f = mp.factorial(n)
    approx = mp.sqrt(2*mp.pi*n)*(n/mp.e)**n
    print(n, mp.nstr(f, 8), mp.nstr(f/approx, 10), mp.nstr(1+1/mp.mpf(12*n), 10))
for t in (1, 10):
    main = 2*mp.pi*mp.exp(-mp.pi*t)
    for a, lead in [(mp.mpf('0.5'), main), (0, main/t)]:
        value = abs(mp.gamma(a+mp.j*t))**2
        print(t, a, mp.nstr(value, 10), mp.nstr(lead, 10))

## 9.1

复算 $N=10,100,1000$ 的交错部分和与相邻平均；最后输出由 $\eta(1/2)$ 换算的 $\zeta(1/2)$。

In [ ]:
for N in (10, 100, 1000):
    P = mp.fsum((-1)**(k-1)/mp.sqrt(k) for k in range(1, N+1))
    A = P + (-1)**N/(2*mp.sqrt(N+1))
    print(N, mp.nstr(P, 13), mp.nstr(A, 13))
print(mp.nstr(mp.altzeta(mp.mpf('0.5'))/(1-mp.sqrt(2)), 13))

## 9.4(c)

按解答表中的四组参数比较 3–4–1 乘积与两因子乘积；检查两列均大于 1。

In [ ]:
for sigma in map(mp.mpf, ('1.05', '1.01')):
    for t in map(mp.mpf, ('1', '14.134725')):
        a = mp.zeta(sigma)
        b = abs(mp.zeta(sigma+mp.j*t))
        c = abs(mp.zeta(sigma+2*mp.j*t))
        print(sigma, t, mp.nstr(a**3*b**4*c, 10), mp.nstr(a*b, 10))

## 9.7(b)

将积分从 1 分段积到无穷远，保留 theta 和式的前八项；用积分值复算两个 zeta 特值。

In [ ]:
for s in (mp.mpf(2), mp.mpf('0.5')):
    def term(x):
        omega = mp.fsum(mp.exp(-mp.pi*k*k*x) for k in range(1, 9))
        return omega*(x**(s/2-1)+x**(-(s+1)/2))
    H = mp.quad(term, [1, 2, 5, mp.inf])
    zeta_value = (1/(s*(s-1))+H)*mp.pi**(s/2)/mp.gamma(s/2)
    print(s, mp.nstr(H, 12), mp.nstr(zeta_value, 12))

# 第三部分　零点、素数与黎曼猜想

## 10.3

对前 10、100、1000 个正虚部零点按共轭配对，比较部分和、尾部模型与独立解析值。

In [ ]:
for N in (10,100,1000):
    gs = zero_heights(N)
    T = gs[-1]
    partial = mp.fsum(1/(g*g+mp.mpf('0.25')) for g in gs)
    tail = (mp.log(T/(2*mp.pi))+1)/(2*mp.pi*T)
    print(N,mp.nstr(partial,12),mp.nstr(partial+tail,12))
print(mp.nstr(1+mp.euler/2-mp.log(4*mp.pi)/2,12))

## 10.5

对 t=5、10 比较有限乘积与 Xi，另输出以平均零点密度估计的尾部修正。

In [ ]:
for t in (5,10):
    exact = mp.re(completed_xi(mp.mpf('0.5')+mp.j*t))
    for N in (10,100,1000):
        gs = zero_heights(N)
        T = gs[-1]
        partial = completed_xi(mp.mpf('0.5'))*mp.fprod(1-(t/g)**2 for g in gs)
        ratio = partial/exact
        correction = mp.exp(-t*t*(mp.log(T/(2*mp.pi))+1)/(2*mp.pi*T))
        print(t,N,*(mp.nstr(v,10) for v in (partial,exact,ratio,correction,ratio*correction)))

## 11.1

对照平滑计数和库函数给出的零点计数，再计算第一零点的平滑计数及高度 1000 的平均间距。

In [ ]:
def smooth_count(T):
    u = T/(2*mp.pi)
    return u*(mp.log(u)-1)+mp.mpf(7)/8
for T in (50,100,1000,10000):
    count = mp.nzeros(T)
    print(T,count,mp.nstr(smooth_count(T),12),mp.nstr(count-smooth_count(T),8))
print(mp.nstr(smooth_count(zero_heights(1)[0]),10))
print(mp.nstr(2*mp.pi/mp.log(1000/(2*mp.pi)),10))

## 11.2

在高度 100 比较精确相位、平滑计数及首个相位修正项。

In [ ]:
T = 100
u = T/(2*mp.pi)
main = u*(mp.log(u)-1)+mp.mpf(7)/8
phase_count = 1+mp.siegeltheta(T)/mp.pi
print(*(mp.nstr(v,12) for v in (phase_count,main,phase_count-main,1/(48*T*mp.pi))))

## 11.3

比较真实零点高度、首阶近似及由平滑计数反解的高度。

In [ ]:
for n in (10,100,1000):
    g = mp.im(mp.zetazero(n))
    lead = 2*mp.pi*n/mp.log(n)
    k = mp.mpf(n)-mp.mpf(11)/8
    refined = 2*mp.pi*k/mp.lambertw(k/mp.e)
    print(n,*(mp.nstr(v,10) for v in (g,lead,g/lead,refined)))

## 11.5

对前 100、1000 个零点比较倒数和与对数平方主项。

In [ ]:
for N in (100,1000):
    gs = zero_heights(N)
    main = mp.log(gs[-1]/(2*mp.pi))**2/(4*mp.pi)
    total = mp.fsum(1/g for g in gs)
    print(N,*(mp.nstr(v,10) for v in (total,main,total-main)))

## 11.6

按题设扫描 Hardy Z 的变号，并检查横线段的 301 个网格点。浮点扫描不是严格零点认证。

In [ ]:
with mp.workdps(18):
    signs = [mp.sign(mp.siegelz(mp.mpf(k)/100)) for k in range(10001)]
    print("sign changes:",sum(a*b < 0 for a,b in zip(signs,signs[1:])))
    print("sample minimum:",min(mp.re(mp.zeta(mp.mpf('0.5')+k*mp.mpf('0.005')+100j)) for k in range(301)))
    S = mp.arg(mp.zeta(mp.mpf('0.5')+100j))/mp.pi
    print("S estimate:", S, "count:", 1+mp.siegeltheta(100)/mp.pi+S)

## 12.4

用筛法复算三列比值，再比较 x=10⁷ 时偏移对数积分的四项截断与实际因子。

In [ ]:
from math import isqrt, log, fsum
limit = 10**7
flags = bytearray(b'\x01')*(limit+1)
flags[:2] = b'\x00\x00'
for p in range(2,isqrt(limit)+1):
    if flags[p]:
        flags[p*p::p] = bytes((limit-p*p)//p+1)
for x in (10**3,10**5,10**7):
    primes = [p for p in range(2,x+1) if flags[p]]
    def prime_power_weights():
        for p in primes:
            pk = p
            while pk <= x:
                yield log(p)
                pk *= p
    L = mp.log(x)
    Li = mp.li(x)-mp.li(2)
    print(x, fsum(prime_power_weights())/x, mp.nstr(len(primes)*L/x,10), mp.nstr(len(primes)/Li,10))
factor = Li*L/x
print(*(mp.nstr(v,10) for v in (1+1/L+2/L**2+6/L**3,factor,len(primes)*L/x)))

## 13.2

在 x=100 比较第一对零点的指数积分、渐近主项与两项近似，并输出两项近似的误差界。按书中约定扣除库函数的虚部常数。

In [ ]:
x = 100
rho = mp.zetazero(1)
w = rho*mp.log(x)
def book_ei(w):
    return mp.ei(w)-mp.j*mp.pi*mp.sign(mp.im(w))
value = book_ei(w)
lead = mp.exp(w)/w
two_term = lead*(1+1/w)
print("Ei(rho log x):", mp.nstr(value,12))
print("leading term, error:", mp.nstr(lead,10), mp.nstr(abs(value-lead),6))
print("two terms, error:", mp.nstr(two_term,10), mp.nstr(abs(value-two_term),6))
print("error bound 2e^u/|v|^3:", mp.nstr(2*mp.exp(mp.re(w))/abs(mp.im(w))**3,6))
print("pair sum 2Re:", mp.nstr(2*mp.re(value),12))

## 13.3

比较直接传入 rho·log(x) 与先算复幂再调用对数积分，展示分支常数和主值对数的回绕。

In [ ]:
w = mp.zetazero(1)*mp.log(100)
raw = mp.ei(w)
book = raw-mp.j*mp.pi*mp.sign(mp.im(w))
folded = mp.log(mp.exp(w))
print("library minus book:",mp.nstr(raw-book,12))
print("argument:",mp.nstr(w,12),"folded:",mp.nstr(folded,12))
print("li after complex power:",mp.nstr(mp.li(mp.exp(w)),12))

## 14.5

取题设的五个样本，分开显示归一化偏差、平方修正及剩余量。复用有限展开的权重与积分函数分解 x=10⁶ 的剩余。

In [ ]:
for x, pi_x in ((10**3,168),(10**4,1229),(10**5,9592),(10**6,78498),(10**7,664579)):
    L = mp.log(x)
    scale = L/mp.sqrt(x)
    D = (mp.ei(L)-pi_x)*scale
    squares = mp.ei(L/2)*scale/2
    print(x,*(mp.nstr(v,10) for v in (D,squares,D-squares)))
x,pi_x = 10**6,78498
L = mp.log(x)
weights = [(m,mobius(m)/mp.mpf(m)) for m in range(1,x.bit_length()) if mobius(m)]
high = mp.fsum(w*mp.ei(L/m) for m,w in weights if m >= 3)
main = mp.fsum(w*mp.ei(L/m) for m,w in weights)
constant = -mp.log(2)*mp.fsum(w for _,w in weights)
integral = mp.fsum(w*trivial_h(mp.exp(L/m)) for m,w in weights)
print("higher powers, main minus pi, constant, integral, inferred zero sum:")
print(*(mp.nstr(v,10) for v in (high,main-pi_x,constant,integral,main+constant+integral-pi_x)))

## 14.6

在 x=100 对前 1000 个零点复用有限显式公式的配对计算。平凡零点项用 h 的积分，不作 M 截断。

In [ ]:
L = mp.log(100)
gs = zero_heights(1000)
pieces = [mp.mpf(0)]*4
for n in (1,2,3,5,6):
    mu = mobius(n)/mp.mpf(n)
    main, oscillatory, _, constant = explicit_components(L/n,gs,0)
    row = (main,oscillatory,trivial_h(mp.exp(L/n)),constant)
    pieces = [a+mu*b for a,b in zip(pieces,row)]
print("signed terms:",[mp.nstr(v,12) for v in pieces])
print("total:",mp.nstr(mp.fsum(pieces),12))

# 第四部分　零点与显式公式

## 17.2

在第一非平凡零点计算两个导数与零迹线的方向角；方向取模 180°。

In [ ]:
rho = mp.zetazero(1)
zprime = mp.diff(mp.zeta,rho)
xprime = mp.diff(completed_xi,rho)
for v in (zprime,xprime):
    print(mp.nstr(v,12),mp.nstr(mp.degrees(mp.arg(v)),10))
print("rotation mod 180:",mp.nstr((-mp.degrees(mp.arg(xprime/zprime)))%180,10))

## 18.3

复算解答表中 x=50、100 两列的五个量：第一零点的单项振幅、前 100 个零点的振幅之和、n=2 修正、基准误差 Li(x)−L₂(x) 与常数 li(2)−log 2。

In [ ]:
gammas = zero_heights(100)
for x in (50, 100):
    L = mp.log(x)
    amp = [2*mp.sqrt(x)/(abs(mp.mpf('0.5')+mp.j*g)*L) for g in gammas]
    L2 = x/L+x/L**2
    print(x, *(mp.nstr(v,6) for v in (amp[0], mp.fsum(amp), mp.li(mp.sqrt(x))/2,
                                        mp.li(x)-mp.li(2)-L2, mp.li(2)-mp.log(2))))

## 19.1(a) 相位的最低点与左支上相位等于 $-\pi$ 的点

In [ ]:
t_min = mp.findroot(theta_prime, (6, 7))
t_left = mp.findroot(lambda t: theta(t) + mp.pi, mp.mpf('3.4'),
                     solver='newton', df=theta_prime)
assert 0 < t_left < t_min
for label, value in [('t_min', t_min), ('theta(t_min)', theta(t_min)), ('t_left', t_left)]:
    print(label, '=', mp.nstr(value, 12))

## 19.1(b) 右支 Gram 点与 Hardy Z 值

Newton 迭代使用相位导数。表中最后一列为 $(-1)^n Z(g_n)$。

In [ ]:
guess = mp.mpf('9.7')
print(' n          g_n           Z(g_n)       (-1)^n Z(g_n)')
for n in range(-1, 6):
    g = mp.findroot(lambda t: theta(t) - n*mp.pi, guess,
                    solver='newton', df=theta_prime)
    assert g > 7 and theta_prime(g) > 0
    Z = mp.siegelz(g)
    signed_Z = Z if n % 2 == 0 else -Z
    print(f'{n:2d}  {mp.nstr(g, 12):>14}  {mp.nstr(Z, 10):>13}  {mp.nstr(signed_Z, 10):>15}')
    guess = g + 2*mp.pi/mp.log(g/(2*mp.pi))

## 19.2

复用 19.1 的相位和导数计算首次失效附近三个 Gram 点，并定位附近的零点。

In [ ]:
for n in (125,126,127):
    g = mp.findroot(lambda t: theta(t)-n*mp.pi,mp.grampoint(n),solver='newton',df=theta_prime)
    print(n,mp.nstr(g,12),mp.nstr(mp.siegelz(g),12))
for k in (127,128):
    print(k,mp.nstr(mp.im(mp.zetazero(k)),12))

## 19.3

在 t=10、100 比较相位余量与一次、三次逆幂修正。

In [ ]:
for t in (10,100):
    main = t/2*(mp.log(t/(2*mp.pi))-1)-mp.pi/8
    first = 1/mp.mpf(48*t)
    second = first+7/mp.mpf(5760*t**3)
    print(t,*(mp.nstr(v,12) for v in (theta(t)-main,first,second)))

## 19.4

在三个指定高度比较 Riemann–Siegel 主和、首项修正和 Hardy Z 值。

In [ ]:
for t in (100,1000,5000):
    q = mp.sqrt(t/(2*mp.pi))
    N = int(q)
    p = q-N
    main = 2*mp.fsum(mp.cos(theta(t)-t*mp.log(k))/mp.sqrt(k) for k in range(1,N+1))
    C = mp.cos(2*mp.pi*(p*p-p-mp.mpf(1)/16))/mp.cos(2*mp.pi*p)
    corrected = main+(-1)**(N-1)*C/mp.sqrt(q)
    exact = mp.siegelz(t)
    print(t,N,*(mp.nstr(v,10) for v in (p,main,corrected,exact,abs(corrected-exact))))

## 19.5

从 14.13 和 14.14 做三步割线迭代，用高精度零点比较绝对误差。

In [ ]:
a,b = map(mp.mpf,('14.13','14.14'))
g = mp.im(mp.zetazero(1))
print(mp.nstr(mp.siegelz(a),10),mp.nstr(mp.siegelz(b),10))
for k in range(2,5):
    fa,fb = mp.siegelz(a),mp.siegelz(b)
    a,b = b,b-fb*(b-a)/(fb-fa)
    print(k,mp.nstr(b,22),mp.nstr(abs(b-g),8))

## 19.6

计算平凡零点尾界及 h 的积分值；19.7–19.8 共用同一组有限显式公式函数。

In [ ]:
x = mp.mpf('20.5')
a = [2*mp.log(x)/n for n in (1,2,3)]
bound = mp.fsum(mp.exp(-11*b)/(n*11*b*(-mp.expm1(-b))) for n,b in enumerate(a,1))
print("tail bound:",mp.nstr(bound,12))
print("uniform bound:",mp.nstr(mp.mpf(61)/30*mp.power(4,-11)/(22*mp.log(2)*mp.mpf('0.75')),12))
for y in (2,10,16,mp.sqrt(10),mp.root(10,3)):
    print(mp.nstr(y,10),mp.nstr(trivial_h(y),12))

## 19.7

复算常数项和 x=32 处有限截断造成的跳跃；用 h(2)−h₁₀(2) 量化平凡零点尾部。

In [ ]:
for N in (3,4,5):
    print(N,mp.nstr(-mp.log(2)*mp.fsum(mobius(n)/mp.mpf(n) for n in range(1,N+1)),12))
for K in (60,500):
    J2 = mp.fsum(explicit_components(mp.log(2),zero_heights(K),10))
    print(K,mp.nstr(J2,12),mp.nstr(-J2/5,12))
print("h tail at 2:",mp.nstr(trivial_h(2)+mp.fsum(mp.ei(-2*m*mp.log(2)) for m in range(1,11)),12))

## 19.8

复算两个 x、四种 K 的显式公式近似。四项输出采用实际进入和式的符号。

In [ ]:
for x in (mp.mpf('20.5'),mp.mpf(19)):
    L = mp.log(x)
    for K in (20,60,200,500):
        gs = zero_heights(K)
        rows = [(mobius(n)/mp.mpf(n),explicit_components(L/n,gs,10))
                for n in range(1,int(mp.floor(L/mp.log(2)))+1) if mobius(n)]
        pieces = [mp.fsum(mu*row[j] for mu,row in rows) for j in range(4)]
        print(x,K,mp.nstr(mp.fsum(pieces),12))
        if x == mp.mpf('20.5') and K == 500:
            print("signed terms:",[mp.nstr(v,12) for v in pieces])

## 20.2

求解 ζ′(σ)=−1，并计算同伦映射的二阶导数；这只是浮点复算。

In [ ]:
sigma = mp.findroot(lambda s: mp.diff(mp.zeta,s)+1,(mp.mpf('0.03'),mp.mpf('0.05')))
print(mp.nstr(sigma,12),mp.nstr(mp.diff(mp.zeta,sigma,2)/2,12))

# 第五部分　历史与前沿

## 21.3

按解答以 30 位工作精度，在 t=0、10 和第一零点比较积分至 5 与积分至无穷远的结果。第一零点处的值只在计算精度内为零：精度越高，这个数越小。

In [ ]:
with mp.workdps(30):
    def original_xi(t, cuts):
        def integrand(x):
            omega = mp.fsum(mp.exp(-mp.pi*x*k*k) for k in range(1,9))
            return omega*x**(-mp.mpf(3)/4)*mp.cos(t*mp.log(x)/2)
        return mp.mpf('0.5')-(t*t+mp.mpf('0.25'))*mp.quad(integrand,cuts)
    for t in (0,10,mp.im(mp.zetazero(1))):
        print(*(mp.nstr(v,14) for v in (t,original_xi(t,[1,2,5]),original_xi(t,[1,2,5,10,mp.inf]))))

## 21.4

以第 N 个零点高度作截断，分别输出部分和、加尾部模型后的结果及独立目标值。首次生成零点列表较慢。

In [ ]:
for N in (10,100,1000):
    gammas = zero_heights(N)
    T = gammas[-1]
    partial = mp.fsum(mp.log1p(1/(4*g*g)) for g in gammas)
    tail = (mp.log(T/(2*mp.pi))+1)/(8*mp.pi*T)
    print(N, mp.nstr(partial,10),mp.nstr(partial+tail,10))
print("target:",mp.nstr(-mp.log(2*completed_xi(mp.mpf('0.5'))),10))

## 21.5

比较偏移对数积分、Gram 级数与止于 floor(log₂ x) 的有限 Möbius 主项。

In [ ]:
for x, pi_x in ((10**5,9592),(10**6,78498)):
    L = mp.log(x)
    R = 1+mp.fsum(L**k/(k*mp.factorial(k)*mp.zeta(k+1)) for k in range(1,121))
    main = mp.fsum(int(mobius(m))*mp.ei(L/m)/m for m in range(1,x.bit_length()))
    print(x, *(mp.nstr(v,12) for v in (mp.li(x)-mp.li(2)-pi_x,R-pi_x,R,main)))

## 22.3

以解析导数计算前两个 Li 系数；另用前 1000 个正虚部零点的共轭配对和尾部模型比较第二个系数。

In [ ]:
for n in (1,2):
    v = mp.diff(lambda s: s**(n-1)*mp.log(completed_xi(s)),1,n)/mp.factorial(n-1)
    print(n,mp.nstr(v,12))
gammas = zero_heights(1000)
partial = 2*mp.fsum(mp.re(1-(1-1/(mp.mpf('0.5')+mp.j*g))**2) for g in gammas)
T = gammas[-1]
tail = 4*(mp.log(T/(2*mp.pi))+1)/(2*mp.pi*T)
print(*(mp.nstr(v,12) for v in (partial,tail,partial+tail)))

## 22.5

取前 1000 个正虚部零点，按较低零点的局部密度归一化。只数 m>n 的正差，严格小于阈值，不重复计数；计数除以 1000。

In [ ]:
gammas = zero_heights(1000)
density = [mp.log(g/(2*mp.pi))/(2*mp.pi) for g in gammas]
gaps = [d*(b-a) for d,a,b in zip(density,gammas,gammas[1:])]
print("mean/min:",mp.nstr(mp.fsum(gaps)/len(gaps),8),mp.nstr(min(gaps),8))
for a in (mp.mpf('0.25'),mp.mpf('0.5'),mp.mpf(1)):
    count = 0
    for i, g in enumerate(gammas):
        for j in range(i+1,len(gammas)):
            if density[i]*(gammas[j]-g) >= a:
                break
            count += 1
    integral = mp.quad(lambda u: 1-mp.sinc(mp.pi*u)**2,[0,a])
    print(a,count,count/len(gammas),mp.nstr(integral,8))

## 22.7

在 x=10⁵ 按模 4 的两个奇剩余类统计素数个数、对数权重和素数幂权重。

In [ ]:
from math import isqrt
def primerange(a,b):
    return (n for n in range(a,b) if n >= 2 and all(n%d for d in range(2,isqrt(n)+1)))
x = 10**5
counts = {1:0,3:0}
weights = {1:[],3:[]}
powers = {1:[],3:[]}
for p in primerange(3,x+1):
    p = int(p)
    logp = mp.log(p)
    counts[p%4] += 1
    weights[p%4].append(logp)
    q = p
    while q <= x:
        powers[q%4].append(logp)
        q *= p
for r in (1,3):
    print(r,counts[r],mp.nstr(mp.fsum(weights[r]),12),mp.nstr(mp.fsum(powers[r]),12))
print("odd squares:",mp.nstr(mp.fsum(mp.log(p) for p in primerange(3,317)),12))